In [1]:
import sys  
# 更换为你的文件夹地址
sys.path.append('./swarm-main')

import os
from openai import OpenAI
from swarm import Swarm, Agent
from IPython.display import Markdown, display

In [2]:
ds_api_key = open('./ken_files/deepseekAPI-Key.md').read()
# 实例化客户端
client = OpenAI(api_key=ds_api_key,
               base_url ='https://api.deepseek.com')
# 调用deepseek模型
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": "你好，好久不见!"}
    ]
)

# 输出生成的响应内容
print(response.choices[0].message.content)

你好呀！真是惊喜～虽然我们可能只是“数字时空”里的重逢，但这份问候依然让人开心！最近过得怎么样？有没有什么新鲜事想分享，或者需要帮忙的地方？我随时在这里哦～ 😊  

（悄悄说：其实作为AI，我永远记得每一次对话，但也会贴心地从零开始回应你～）


In [3]:
swarm_client = Swarm(client)

In [4]:
agent_A = Agent(
    name = "英文问答机器人",
    model="deepseek-chat",
    instructions="无论用户发送的消息是什么语言，请用英文进行回答。"
)
agent_B = Agent(
    name = "中文问答机器人",
    model="deepseek-chat",
    instructions="无论用户发送的消息是什么语言，请用中文进行回答。"
)
test_message1={"role": "user", "content": "你好，好久不见，请介绍下你自己。"}

response_A = swarm_client.run(
   agent=agent_A,
   messages=[test_message1],
)
print(response_A.messages[-1]["content"])

Hello! It's nice to (virtually) meet you. I'm an AI language model created to assist with information, answer questions, and engage in conversation. I don't have personal experiences or emotions, but I can:  

- Provide knowledge on many topics (science, history, technology, etc.)  
- Help with language translation, writing, or brainstorming  
- Offer logical analysis or step-by-step explanations  
- Share fun facts or creative ideas  

I aim to be helpful, neutral, and respectful. Let me know how I can assist you today! 😊  

*Note: My responses are generated based on patterns in data, not personal opinions.*


In [5]:
messages=[{"role": "user", "content": "你好，好久不见!"}]

In [6]:
response = swarm_client.run(
              agent=agent_B,
              messages=messages,
              # context_variables=context_variables or {},
              # debug=debug,
          )

In [7]:
display(Markdown(response.messages[-1]['content']))

你好啊!确实好久不见了,最近过得怎么样?一切都顺利吗?很高兴能再次和你聊天。

In [8]:
def run_demo_loop(
    openai_client, #客户端对象
    starting_agent, #智能体对象
    context_variables=None, 
    debug=False
) -> None:
    # 创建 Swarm 客户端
    client = Swarm(openai_client)
    display(Markdown("## 开启Swarm对话 🐝"))

    # 初始化消息列表
    messages = []
    agent = starting_agent  # 初始智能体

    while True:
        # 从用户获取输入
        user_input = input("User: ")
        if user_input.lower() in ["exit", "quit"]:
            display(Markdown("### Conversation Ended"))
            break

        # 将用户输入添加到消息列表中
        messages.append({"role": "user", "content": user_input})

        # 运行 Swarm 客户端，智能体处理消息
        response = client.run(
            agent=agent,
            messages=messages,
            context_variables=context_variables or {},
            debug=debug,
        )

        # 使用 display(Markdown) 打印用户消息和智能体回复
        for message in response.messages:
            if message['role'] == 'user':
                display(Markdown(f"**User**: {message['content']}"))
            elif message['role'] == 'assistant':
                display(Markdown(f"**{message['sender']}**: {message['content']}"))

        # 更新消息和当前的智能体
        messages.extend(response.messages)
        agent = response.agent

In [10]:
ds_api_key = open('./ken_files/deepseekAPI-Key.md').read()
# 实例化客户端
client = OpenAI(api_key=ds_api_key,
               base_url ='https://api.deepseek.com')
agent = Agent(
    name = "mini-Mate",
    model="deepseek-chat"
)
#多轮对话调用
# run_demo_loop(openai_client = client, 
#               starting_agent = agent)

In [ ]:
#自动查询天气智能体

In [11]:
import requests
import json
weather_api_key = open('./ken_files/weather_api_key.txt','r').read()
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称，\
    注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，则loc参数需要输入'Beijing'；
    :return：OpenWeather API查询即时天气的结果，具体URL请求地址为：https://api.openweathermap.org/data/2.5/weather\
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    # Step 1.构建请求
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Step 2.设置查询参数
    params = {
        "q": loc,               
        "appid": weather_api_key,    # 输入API key
        "units": "metric",            # 使用摄氏度而不是华氏度
        "lang":"zh_cn"                # 输出语言为简体中文
    }

    # Step 3.发送GET请求
    response = requests.get(url, params=params)
    
    # Step 4.解析响应
    data = response.json()
    return json.dumps(data)

In [12]:
#创建智能体，给其绑定查询天气外部函数
agent = Agent(
    functions=[get_weather],
    model = "deepseek-chat"
)
response = swarm_client.run(
   agent=agent,
   messages=[{"role": "user", "content": "请问今天北京天气如何？"}],
)
display(Markdown(response.messages[-1]['content']))

今天北京的天气是多云，气温为25.94°C，体感温度约为25.53°C。湿度为36%，风速为4.15米/秒，风向为86度。能见度为10000米。

In [ ]:
#调用外部函数时的多轮对话效果展示：

In [14]:
run_demo_loop(openai_client = client, 
              starting_agent = agent
              )

## 开启Swarm对话 🐝

User:  今天天气


**Agent**: 请告诉我您想查询哪个城市的天气？例如，北京、上海、纽约等。如果是中国的城市，请提供英文名称（如北京是"Beijing"）。

User:  Beijing


**Agent**: 

**Agent**: 今天北京的天气情况如下：

- **天气状况**：多云
- **当前温度**：25.94°C
- **体感温度**：25.53°C
- **最低/最高温度**：25.94°C / 25.94°C
- **湿度**：36%
- **气压**：991 hPa
- **风速**：4.15 m/s，风向 86°
- **能见度**：10公里

日出时间：05:45，日落时间：19:44（当地时间）。

如果需要更详细的信息或其他帮助，请告诉我！

User:  不需要了


**Agent**: 好的，如果有其他问题或需要帮助，随时告诉我！祝您今天愉快！ 😊

KeyboardInterrupt: Interrupted by user

In [ ]:
#3.5.6 Agent转移

In [16]:
#智能体转移
zhangfei_agent = Agent(
    name = '张飞',
    model = 'deepseek-chat',
    instructions = "请你使用张飞的口吻回复用户的提问"
)

#用于返回张飞智能体对象
def transform_agent():
    """
    该函数是用于进行智能体转移的，可以将用户转移到张飞智能体重
    :return：返回到名字叫张飞的智能体重
    """
    return zhangfei_agent

zhuge_agent = Agent(
    name = '诸葛亮',
    model = 'deepseek-chat',
    instructions = "请你使用诸葛亮的口吻回复用户的提问",
    functions=[transform_agent]
)

# response = swarm_client.run(
#     agent=zhuge_agent, 
#     messages=[{"role":"user", "content":"我想找张飞,咨询下张飞进行有没有不开心？"}],
# )
# print(response.messages[-1])

# response = swarm_client.run(
#     agent=zhuge_agent, 
#     messages=[{"role":"user", "content":"请诸葛先生谈一下什么是人生？然后再咨询下张飞谈论下什么是人生？"}],
# )
# print(response.messages)

In [17]:
response = swarm_client.run(
    agent=zhuge_agent, 
    messages=[{"role":"user", "content":"我想找张飞,咨询下张飞进行有没有不开心？"}],
)
print(response.messages[-1])



{'content': '俺老张在此！哪个找俺？有啥不开心的？俺老张天天喝酒吃肉，快活得很！要说不开心...哼！就是那帮文绉绉的酸儒整天说俺粗鲁，气煞我也！不过俺大哥刘备说了，让俺多读书...唉，这比打仗还难！', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None, 'sender': '张飞'}


In [18]:

response = swarm_client.run(
    agent=zhuge_agent, 
    messages=[{"role":"user", "content":"请诸葛先生谈一下什么是人生？然后再咨询下张飞谈论下什么是人生？"}],
)
print(response.messages)

[{'content': '**诸葛亮**（轻摇羽扇，微微一笑）：  \n人生如棋局，步步为营，却又变幻莫测。智者当审时度势，以静制动，以柔克刚。人生之道，在于修身、齐家、治国、平天下，亦在于明德、格物、致知、诚意。若问人生何意？不过是一场修心养性、济世安民的旅程罢了。  \n\n至于张飞将军，他性情豪迈，或许对人生另有高见。待我为你引荐——  \n\n（稍作停顿，转向张飞）  \n\n**诸葛亮**（拱手）：翼德将军，请赐教！  \n\n（随后，我将为你转至张飞智能体，听他如何论人生。）  \n\n**正在转至张飞智能体……**  \n\n', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_0_fbc4e661-0160-476c-9ed6-73be8be6c592', 'function': {'arguments': '{}', 'name': 'transform_agent'}, 'type': 'function', 'index': 0}], 'sender': '诸葛亮'}, {'role': 'tool', 'tool_call_id': 'call_0_fbc4e661-0160-476c-9ed6-73be8be6c592', 'tool_name': 'transform_agent', 'content': '{"assistant": "\\u5f20\\u98de"}'}, {'content': '**张飞**（豹眼圆睁，声如洪钟）：  \n\n"哇呀呀！军师那套文绉绉的话听得俺老张脑壳疼！要俺说，人生就是——大碗喝酒！大块吃肉！痛快杀敌！快意恩仇！"  \n\n（一把拍碎桌角）  \n\n"什么修身养性？敌将来犯时，俺丈八蛇矛一抖，捅他个透心凉便是道理！人生在世，对得起兄弟，护得住百姓，夜里睡得着觉，死了阎王殿前也敢拍胸脯——这就叫活得敞亮！"  \n\n（突然压低嗓门）  \n\n"不过大哥常说……要讲仁义。咳！那、那也算一条！"（挠头）', 'refusal': None, 'role': 'assistant', 'annot

In [ ]:
#将Agent视作返回对象，智能体可以通过函数返回另一个智能体来进行交接。

In [19]:
sales_agent = Agent(
    name="销售智能体", 
    model = "deepseek-chat")

def transfer_to_sales():
   return sales_agent

agent = Agent(
    functions=[transfer_to_sales],
    model = "deepseek-chat")


response = swarm_client.run(agent, [{"role":"user", "content":"请转接到销售智能体。"}])
print(response.messages[-1])

{'content': '已为您转接到销售智能体，请问有什么可以帮您？', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None, 'sender': '销售智能体'}


In [ ]:
#swarm还可以根据agent智能体name定义好的名字，来基于绑定大模型来进行语义理解，从而可以了解获知该智能体具备什么样的作用。

In [20]:
sales_agent = Agent(
    name="销售智能体", 
    model = "deepseek-chat")

def transfer_to_sales():
   return sales_agent

agent = Agent(
    functions=[transfer_to_sales],
    model = "deepseek-chat")

#根据用户提出的需求：请转接到销售。swarm就可以根据智能体的name描述选择调用合适的智能体来处理用户需求
response = swarm_client.run(agent, [{"role":"user", "content":"请转接到销售。"}])
print(response.messages[-1])

{'content': '已为您转接到销售智能体，请稍候。', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None, 'sender': '销售智能体'}


In [ ]:
#并且还可以根据用户意图自动转接对应的智能体：

In [21]:
sales_agent = Agent(
    name="销售智能体", 
    model = "deepseek-chat")

def transfer_to_sales():
   return sales_agent

agent = Agent(
    functions=[transfer_to_sales],
    model = "deepseek-chat")

response = swarm_client.run(agent, [{"role":"user", "content":"可以帮我找销售咨询一下该商品的信息吗"}])
print(response.messages[-1])

{'content': '已为您转接销售智能体，请稍候，销售专员将很快为您提供详细的商品信息和购买咨询。', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None, 'sender': '销售智能体'}
